# Free Gaussian electron wave packet

Follow a one-dimensional, initially unchirped Gaussian under the free,
nonrelativistic Schroedinger equation. The initial position standard deviation
is $\sigma_0$, and the mean velocity is $v_0$. There is no external potential.
Run the setup in [examples/README.md](README.md) first.

The probability density remains Gaussian:
$$
\rho(x,t)=\frac{1}{\sqrt{2\pi}\sigma(t)}
\exp\left[-\frac{(x-v_0t)^2}{2\sigma(t)^2}\right],\quad
\sigma(t)=\sigma_0\sqrt{1+(t/\tau)^2},\quad
\tau=\frac{2m\sigma_0^2}{\hbar}.
$$
Width here always means the RMS width of the probability density, not the
width parameter of the wavefunction amplitude. The initial momentum spread
is $\Delta p=\hbar/(2\sigma_0)$ and stays constant during free evolution.

Reference: [MIT 8.04, Lecture 7](https://ocw.mit.edu/courses/8-04-quantum-physics-i-spring-2016/resources/mit8_04s16_lecnotes7/).

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from unitx import Quantity
from unitx.si import SI, si

sigma0 = 10 * si.nm
mean_velocity = 20_000 * si.m / si.s
times = np.asarray([0.0, 2.0, 5.0, 10.0]) * si.ps
x = np.linspace(-400, 800, 1601)[:, None] * si.nm
spreading_time = 2 * SI.me * sigma0**2 / SI.hbar
widths = sigma0 * np.sqrt(1 + (times / spreading_time) ** 2)
centers = mean_velocity * times
rho = np.exp(-((x - centers) ** 2) / (2 * widths**2)) / (np.sqrt(2 * np.pi) * widths)

print('Spreading time:', spreading_time.to(si.ps))
print('RMS widths:', widths)

## Probability density carries its unit

Plot the density and width using their current units, and take each axis
label from the corresponding Quantity. The area under the density, rather
than its height, is the normalization. Shorter initial packets spread faster
because their initial momentum spread is larger.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), layout='constrained')
axes[0].plot(x[:, 0].value, rho.value)
axes[0].set(
    xlabel=f'Position [{x.unit}]',
    ylabel=f'Probability density [{rho.unit}]',
    title='Free propagation',
)
axes[0].legend([f't = {time}' for time in times])
axes[1].plot(times.value, widths.value, 'o-')
axes[1].set(
    xlabel=f'Time [{times.unit}]',
    ylabel=f'RMS width [{widths.unit}]',
    title='Wave-packet spreading',
)
plt.show()

## Check normalization, drift, and the uncertainty product

`np.trapezoid(rho, x=x, axis=0)` integrates the Quantity directly. The
coordinate unit multiplies the density unit, giving a dimensionless
normalization. Position moments then carry length and length-squared units
automatically. The finite grid must cover the tails and resolve the
narrowest packet.

Initially $\sigma_0\Delta p=\hbar/2$. At later times the position width
grows and position-momentum correlations develop; the simple product need
not remain at its initial minimum.

In [ ]:
normalization = np.trapezoid(rho, x=x, axis=0)
mean_position = np.trapezoid(x * rho, x=x, axis=0)
variance = np.trapezoid((x - mean_position) ** 2 * rho, x=x, axis=0)
assert isinstance(normalization, Quantity)
assert isinstance(mean_position, Quantity)
assert isinstance(variance, Quantity)
numerical_width = variance**0.5
assert np.all(normalization.isclose(1, rel_tol=1e-9))
assert np.all(mean_position.isclose(centers, abs_tol=1e-7 * x.unit))
assert np.all(numerical_width.isclose(widths, rel_tol=1e-8))
momentum_spread = SI.hbar / (2 * sigma0)
uncertainty_ratio = widths * momentum_spread / (SI.hbar / 2)
assert uncertainty_ratio[0].isclose(1)
assert np.all(uncertainty_ratio >= 1 - 1e-12)
assert mean_velocity / SI.c < 1e-3
print('Probability integrals:', normalization)
print('Position-momentum product / (hbar/2):', uncertainty_ratio)